# Count the number of colocalization points in the Mediterranean sea
## What is done here ?
- select the drifters points under the swaths for each cycle considering drifters points for which it is the nearest swot image (one cycle every 23h50 approximately, e.g drifters in the $\pm$ 12h25 before and after the swot image)
- compute the minimum/maximum time gap for each swot image/cycle (will depend on the dt of the regular grid)

## Results ?
- some cycle are not available :
    - swath 3 : 498, 526-528, 566, 572
    - swath 16 : 480, 508, 513, 526-528, 534

In [1]:
import numpy as np
import pandas as pd
import xarray as xr

import matplotlib.pyplot as plt

import os
from glob import glob

from cstes import swot_dir, drifters_dir, get_proj, lonlat2xy, zarr_dir
from swot import browse_swot_250m, add_mask_inside_swot, build_swath_polygon

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.geodesic as cgeo
crs = ccrs.PlateCarree()

import cartopy.geodesic as geod
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import pyproj
from rasterio.transform import Affine

import pynsitu as pyn

In [ ]:
def plot_error(df, x, v, ax, suf = 'er__', color ="silver", alpha=1):
    ax.fill_between(
        df[x], df[v] - df[suf + v], df[v] + df[suf + v], color=color, alpha=alpha
    )

________
# CHOOSE PARAMETERS HERE

In [9]:
# TO CHOOSE 
drifters_sources = 'all_med_variational_10min_v1.nc'
spectral_decomp = True
low_pass = False
#dt = '12h' #'nearestswath'
dt = '10d' #'nearestswath'

cutoff = 0.88

_______
# Data 

In [10]:
# SWOT 250m browsing dataframe
dfs = browse_swot_250m().reset_index()
# Drifter 
if spectral_decomp : 
    dr = xr.open_dataset(os.path.join(zarr_dir,'before_coloc','drifters', 'spectral_decomp_'+drifters_sources))
    drifters_sources = 'spectral_decomp_'+drifters_sources

elif low_pass : 
    dr = xr.open_dataset(os.path.join(zarr_dir,'before_coloc','drifters',f"low_pass_{str(cutoff).replace('0.', '0')}_"+drifters_sources))
    drifters_sources = f"low_pass_{str(cutoff).replace('0.', '0')}_"+drifters_sources
    
else : 
    dr = xr.open_dataset(os.path.join(drifters_dir,'L2', drifters_sources))

dr_key = dr[['cruise_id', 'drifter_type']]
dr = dr.where(dr.gap_mask==1)

_____
# Select drifters under SWOT swath

In [11]:
def sel_drifters(dr, dt, swath, cycle):
    dfs_ = dfs.where((dfs.pass_number==swath)&(dfs.cycle_number==cycle)).dropna()
    dss = xr.open_dataset(dfs.where((dfs.pass_number==swath)&(dfs.cycle_number==cycle)).dropna().file.values[0])
    
    #time
    if dt == 'nearestswath':
        dr_ = dr.sel(datetime = slice(pd.to_datetime(dfs_.start_time_cut).values[0], pd.to_datetime(dfs_.end_time_cut).values[0]))
    else : 
        tmin = (pd.to_datetime(dfs_.time)-pd.Timedelta(dt)).values[0]
        tmax = (pd.to_datetime(dfs_.time)+pd.Timedelta(dt)).values[0]
        dr_ = dr.sel(datetime = slice(tmin, tmax))

    dr_= add_mask_inside_swot(dss, dr_)

    #under swath
    dr_ = dr_.where(dr_["inside_left"]+dr_["inside_right"])
    dfr_ = dr_.to_dataframe().reset_index().dropna()
    #time to nearest swot
    dfr_['time_to_swot']=(dfr_.datetime-dfs_.time.values[0]).abs()

    # stats for each drifters
    dfrs_=pd.DataFrame()
    dfrs_['time_to_swot_min'] = dfr_.groupby('drifter_id').time_to_swot.min()
    dfrs_['time_to_swot_max'] = dfr_.groupby('drifter_id').time_to_swot.max()
    dfrs_['point_number'] = dfr_.groupby('drifter_id').datetime.count()
    dfrs_['cycle_number'] = int(dfs_.cycle_number.values[0])
    dfrs_['cycle_date'] = dfs_.time.values[0]
    dfrs_['pass_number'] = int(dfs_.pass_number.values[0])
    return dfs_, dfr_, dfrs_.reset_index()

In [12]:
# EXAMPLE
dfs_, dfr_, dfrs_ = sel_drifters(dr, '12h', 3, 480)
dfr_

/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'netcdf4' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)
/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'scipy' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)


,drifter_id,datetime,x,y,cruise_id,lonc,latc,longitude,latitude,velocity_east,...,diurnal_acceleration_east,diurnal_acceleration_north,semidiurnal_velocity_east,semidiurnal_velocity_north,semidiurnal_acceleration_east,semidiurnal_acceleration_north,inside_left,inside_right,drifter_type,time_to_swot
288,0-4388599,2023-04-03 11:30:00,-204778.730979,216967.924020,C-SWOT,7.317303,39.945436,4.850667,41.873353,0.066847,...,0.000004,0.000004,0.012206,0.000080,-4.805423e-07,-2.130806e-06,1.0,0.0,CARTHE,0 days 11:53:48.189185728
289,0-4388599,2023-04-03 11:40:00,-204736.781430,216780.776745,C-SWOT,7.317303,39.945436,4.851236,41.871680,0.073324,...,0.000004,0.000004,0.011873,-0.001205,-6.291371e-07,-2.145342e-06,1.0,0.0,CARTHE,0 days 11:43:48.189185728
290,0-4388599,2023-04-03 11:50:00,-204690.742692,216595.661729,C-SWOT,7.317303,39.945436,4.851853,41.870026,0.080445,...,0.000004,0.000004,0.011451,-0.002495,-7.741656e-07,-2.143830e-06,1.0,0.0,CARTHE,0 days 11:33:48.189185728
291,0-4388599,2023-04-03 12:00:00,-204640.247170,216412.223851,C-SWOT,7.317303,39.945436,4.852524,41.868388,0.088115,...,0.000004,0.000004,0.010944,-0.003778,-9.145439e-07,-2.126115e-06,1.0,0.0,CARTHE,0 days 11:23:48.189185728
292,0-4388599,2023-04-03 12:10:00,-204585.004972,216230.012026,C-SWOT,7.317303,39.945436,4.853251,41.866762,0.096177,...,0.000004,0.000003,0.010354,-0.005046,-1.049214e-06,-2.092167e-06,1.0,0.0,CARTHE,0 days 11:13:48.189185728
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17419,0-4388557,2023-04-04 10:40:00,-164126.049301,33013.476714,C-SWOT,7.737819,42.112607,5.744363,42.392547,-0.134644,...,0.000005,0.000002,0.013890,-0.019266,1.783363e-09,-1.900542e-06,0.0,1.0,CARTHE,0 days 11:16:11.810814272
17420,0-4388557,2023-04-04 10:50:00,-164205.445170,33030.421909,C-SWOT,7.737819,42.112607,5.743394,42.392683,-0.130309,...,0.000005,0.000002,0.013837,-0.020333,-1.774194e-07,-1.649424e-06,0.0,1.0,CARTHE,0 days 11:26:11.810814272
17421,0-4388557,2023-04-04 11:00:00,-164282.420287,33050.994807,C-SWOT,7.737819,42.112607,5.742454,42.392852,-0.126491,...,0.000005,0.000001,0.013677,-0.021246,-3.543223e-07,-1.386185e-06,0.0,1.0,CARTHE,0 days 11:36:11.810814272
17422,0-4388557,2023-04-04 11:10:00,-164357.234470,33074.600624,C-SWOT,7.737819,42.112607,5.741539,42.393049,-0.123035,...,0.000005,0.000001,0.013412,-0.021997,-5.275565e-07,-1.112825e-06,0.0,1.0,CARTHE,0 days 11:46:11.810814272


In [6]:
os.path.join(zarr_dir,'coloc_files', 'drifters', f'drifterscoloc_{dt}_'+drifters_sources.replace('.nc', '.csv'))

'/home/datawork-lops-oc/aponte/margot/DATA_MED_COLOC_v2.0.1/coloc_files/drifters/drifterscoloc_12h_spectral_decomp_all_med_variational_10min_v1.csv'

_____
# Create and store dataset

In [ ]:
D = []
for swath in [3,16]:
    for cycle in dfs.where(dfs.pass_number==swath).dropna().cycle_number:
        dfs_, dfr_, dfrs_ = sel_drifters(dr, dt, swath, cycle)
        dfr_['cycle_number'] = int(dfs_.cycle_number.values[0])
        dfr_['cycle_date'] = dfs_.time.values[0]
        dfr_['pass_number'] = int(dfs_.pass_number.values[0])
        D.append(dfr_)
        print(cycle)
        
df = pd.concat(D).set_index('pass_number').reset_index()
df['row_number'] = np.arange(len(df))
df = df.set_index('row_number')

# Store
df.to_csv(os.path.join(zarr_dir,'coloc_files', 'drifters', f'drifterscoloc_{dt}_'+drifters_sources.replace('.nc', '.csv')))

/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'netcdf4' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)
/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'scipy' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)


474.0


/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'netcdf4' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)
/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'scipy' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)


475.0


/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'netcdf4' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)
/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'scipy' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)


...

575.0


/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'netcdf4' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)
/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'scipy' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)


576.0


/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'netcdf4' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)
/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'scipy' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)


577.0


In [16]:
os.path.join(zarr_dir,'coloc_files', 'drifters', f'drifterscoloc_{dt}_'+drifters_sources.replace('.nc', '.csv'))

'/home/datawork-lops-oc/aponte/margot/DATA_MED_COLOC_v2.0.1/coloc_files/drifters/drifterscoloc_10d_spectral_decomp_all_med_variational_10min_v1.csv'

In [14]:
D = []
for swath in [3,16]:
    for cycle in dfs.where(dfs.pass_number==swath).dropna().cycle_number:
        dfs_, dfr_, dfrs_ = sel_drifters(dr, dt, swath, cycle)
        D.append(dfrs_)
df_stats = pd.concat(D)
df_stats.to_csv(os.path.join(zarr_dir,'drifters', f'driftersstats_{dt}_'+drifters_sources.replace('.nc', '.csv')))

/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'netcdf4' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)
/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'scipy' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)
/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'netcdf4' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)
/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'scipy' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)
/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/xarray/backends/plugins.py:159: Runt

In [7]:
df

,pass_number,drifter_id,datetime,x,y,cruise_id,lonc,latc,longitude,latitude,...,semidiurnal_velocity_east,semidiurnal_velocity_north,semidiurnal_acceleration_east,semidiurnal_acceleration_north,inside_left,inside_right,drifter_type,time_to_swot,cycle_number,cycle_date
row_number,,,,,,,,,,,,,,,,,,,,,
0,3,300534060315840,2023-04-01 11:50:00,-98662.911488,19783.062142,C-SWOT,6.0070,42.0490,4.811970,42.220893,...,0.009955,-0.009880,-2.326338e-07,-0.000002,1.0,0.0,SVP,0 days 11:52:32.696536,478,2023-04-01 23:42:32.696536000
1,3,300534060315840,2023-04-01 12:00:00,-98623.960276,19842.195526,C-SWOT,6.0070,42.0490,4.812432,42.221430,...,0.009778,-0.011003,-3.583966e-07,-0.000002,1.0,0.0,SVP,0 days 11:42:32.696536,478,2023-04-01 23:42:32.696536000
2,3,300534060315840,2023-04-01 12:10:00,-98583.178329,19897.134982,C-SWOT,6.0070,42.0490,4.812916,42.221930,...,0.009525,-0.012047,-4.817391e-07,-0.000002,1.0,0.0,SVP,0 days 11:32:32.696536,478,2023-04-01 23:42:32.696536000
3,3,300534060315840,2023-04-01 12:20:00,-98541.006388,19948.108049,C-SWOT,6.0070,42.0490,4.813418,42.222394,...,0.009200,-0.013003,-6.017149e-07,-0.000002,1.0,0.0,SVP,0 days 11:22:32.696536,478,2023-04-01 23:42:32.696536000
4,3,300534060315840,2023-04-01 12:30:00,-98497.853233,19995.403935,C-SWOT,6.0070,42.0490,4.813933,42.222825,...,0.008803,-0.013863,-7.174011e-07,-0.000001,1.0,0.0,SVP,0 days 11:12:32.696536,478,2023-04-01 23:42:32.696536000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
376718,16,300534060015760,2023-07-10 06:10:00,-145207.110838,-28177.646918,C-SWOT,4.1131,40.0546,2.417786,39.788394,...,-0.040939,-0.009495,-1.738689e-06,0.000005,1.0,0.0,SVP,0 days 11:13:38.131544384,577,2023-07-09 18:56:21.868455616
376719,16,300534060015760,2023-07-10 06:20:00,-145283.518098,-28121.682437,C-SWOT,4.1131,40.0546,2.416881,39.788885,...,-0.041829,-0.006689,-1.221971e-06,0.000005,1.0,0.0,SVP,0 days 11:23:38.131544384,577,2023-07-09 18:56:21.868455616
376720,16,300534060015760,2023-07-10 06:30:00,-145362.329231,-28063.105917,C-SWOT,4.1131,40.0546,2.415948,39.789399,...,-0.042406,-0.003842,-6.963034e-07,0.000005,1.0,0.0,SVP,0 days 11:33:38.131544384,577,2023-07-09 18:56:21.868455616
